# QS-ENN — Aggregate Analysis

This notebook aggregates the fold-level outputs from the ten dataset notebooks. It produces dataset summaries, cross-dataset performance tables, ablation summaries, traceability checks, and applicability diagnostics.


## 1. Environment setup


In [ ]:
# Google Colab setup
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Update these paths for your environment.
PROJECT_DIR = "/content/drive/MyDrive/DRIVES3/qsenn_project/"
RAW_DATA_DIR = "/content/drive/MyDrive/DRIVES3/DATA/"

!pip -q install pennylane imbalanced-learn 2>/dev/null

%run {PROJECT_DIR}qsenn.py
init(data_dir=RAW_DATA_DIR, out_dir=PROJECT_DIR, profile="medium")


## 2. Experiment status


In [ ]:
import glob
import os
import pandas as pd

completed = sorted(
    os.path.basename(f)[6:-4]
    for f in glob.glob(outp("folds_*.csv"))
    if "ALL" not in f
)
print(f"{len(completed)}/{len(REGISTRY)} datasets completed: {completed}")

missing = [key for key in REGISTRY if key not in completed]
if missing:
    print(f"Missing: {missing}")


## 3. Dataset and preprocessing summary

Retained variance is reported as mean ± standard deviation across folds because PCA is fitted independently within each training fold.


In [ ]:
table1 = table_one_full()
display(table1)
table1.to_csv(outp("TABLE1.csv"), index=False)


## 4. Performance Tables 2, 3, and 5

All deltas are computed from unrounded paired fold-level values. The traceability check verifies internal consistency between the reported cross-dataset means and paired differences.


In [ ]:
all_folds = load_all_folds()
table2, table3, table5, traceability = make_tables(all_folds)

display(table2.round(3))
display(table3.round(3))
display(table5.round(3))

if len(traceability):
    print(f"Maximum traceability difference: {traceability['difference'].max():.2e}")


## 5. Cross-dataset ablation summary (Table 4)

Sign convention: **Δ = ablated variant − complete QS-ENN**.


In [ ]:
ablation_files = sorted(glob.glob(outp("ablation_*.csv")))

if ablation_files:
    ablation_all = pd.concat([pd.read_csv(f) for f in ablation_files], ignore_index=True)
    table4 = (
        ablation_all[ablation_all["metric"] == "F1_1"]
        .pivot_table(index="dataset", columns="variant", values="delta")
        .round(4)
    )
    display(table4)
    table4.to_csv(outp("TABLE4.csv"))
else:
    print("No ablation files were found.")


## 6. Applicability diagnostics

Simple quantum-geometry descriptors are evaluated using leave-one-dataset-out sign prediction and Spearman rank correlation. These diagnostics are exploratory and should be interpreted in light of the small number of datasets.


In [ ]:
applicability = applicability_lodo()
if applicability is not None:
    display(applicability.round(4))


## 7. Cross-dataset F1 summary


In [ ]:
f1_summary = all_folds.pivot_table(
    index="dataset",
    columns="arm",
    values="F1_1"
).round(3)
display(f1_summary)


## 8. Optional equal-budget nested-CV summary

This section summarizes nested-CV output files when they are available.


In [ ]:
nested_files = [
    f for f in sorted(glob.glob(outp("nested_*.csv")))
    if "chosen" not in os.path.basename(f)
]

if nested_files:
    nested = pd.concat([pd.read_csv(f) for f in nested_files], ignore_index=True)

    fixed_qsenn = (
        all_folds[all_folds["arm"] == "QKNN + QS-ENN"]
        .groupby("dataset")["F1_1"].mean()
        .rename("Fixed F1")
    )
    nested_qsenn = (
        nested[nested["arm"].isin(["QS-ENN", "QKNN + QS-ENN"])]
        .groupby("dataset")["F1_1"].mean()
        .rename("Nested F1")
    )

    nested_summary = pd.concat([fixed_qsenn, nested_qsenn], axis=1)
    display(nested_summary.round(3))
else:
    print("No nested-CV result files were found.")
